<font size="7">
Analysis of test directories
</font>

In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO

In [65]:
def filter_comments(file_path):
    """
    Generator to read a file and clubs lines that do not start 
    with '#' or '@'.
    """
    not_commented = []
    with open(file_path, 'r') as f:
        for line in f:
            if not line.strip().startswith('#') and not line.strip().startswith('@'):
                not_commented.append(line)
    return not_commented


# thermodynamic properties

## equilibration
After equilibration (or during) check the thermodynamic property fluctuations through ` mpirun gmx energy -f npt_eq.edr -o properties/thermo.xvg` and selecting these *7 8 9 11 12 17 39* choices.

In [66]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/test/equilibration/Test-5/surfactant_water/equilibration/npt'

In [67]:
file = 'properties/thermo.xvg'
# Use StringIO to treat the filtered lines as a file for read_csv
removed_comments = filter_comments(f'{dir}/{file}')
file_removed_comments = StringIO('\n'.join(removed_comments))
df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=['time (ps)', 'potential', 'kinetic', 'total E', 'T', 'P', 'density', 'system-T'])

In [ ]:
df.head()

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['potential'], label='PE')
ax.plot(df['time (ps)'], df['kinetic'], label='KE')
ax.plot(df['time (ps)'], df['total E'], label='TE')
ax.set(xlabel= 'Time (ps)', ylabel='Energy (kJ/mol)')
ax.legend()

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['density'], label='PE')
ax.set(xlabel= 'Time (ps)', ylabel='Density (kg/m^3)')

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['P'])
ax.axhline(1, ls='--', color='black')
ax.set(xlabel= 'Time (ps)', ylabel='Pressure (bar)')

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['T'])
ax.axhline(298, ls='--', color='black')
ax.set(xlabel= 'Time (ps)', ylabel='Temperature (K)')

## production
During production check the thermodynamic property fluctuations through `mpirun gmx energy -f npt_eq.edr -o properties/thermo.xvg` and selecting these *7 8 9 11 12 32* choices.

In [73]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/test/equilibration/Test-5/surfactant_water/production/nvt'

In [74]:
file = 'properties/thermo.xvg'
# Use StringIO to treat the filtered lines as a file for read_csv
removed_comments = filter_comments(f'{dir}/{file}')
file_removed_comments = StringIO('\n'.join(removed_comments))
df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=['time (ps)', 'potential', 'kinetic', 'total E', 'T', 'P', 'system-T'])

In [ ]:
df.head()

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['potential'], label='PE')
ax.plot(df['time (ps)'], df['kinetic'], label='KE')
ax.plot(df['time (ps)'], df['total E'], label='TE')
ax.set(xlabel='Time (ps)', ylabel='Energy (kJ/mol)')
ax.legend()

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['P'])
ax.set(xlabel='Time (ps)', ylabel='Pressure (bar)')

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['T'])
ax.axhline(298, ls='--', color='black')
ax.set(xlabel='Time (ps)', ylabel='Tempearture (K)')

# structural properties

In [79]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/test/equilibration/Test-5/surfactant_water/production/nvt'

## rdf
rdf between tail and one head group of the surfactant. average over last set of timeframes: `mpirun gmx rdf -f nvt_prod.xtc -s nvt_prod.tpr -n custom.ndx -ref Tail -sel Head_1_2 -o properties/rdf_head12_tail.xvg -bin 0.05 -b 968000 -e 1000000` where `-b` and `-e` denotes begining and ending time (ps) to average the rdf.

In [80]:
file = 'properties/rdf_head12_tail.xvg'
# Use StringIO to treat the filtered lines as a file for read_csv
removed_comments = filter_comments(f'{dir}/{file}')
file_removed_comments = StringIO('\n'.join(removed_comments))
df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=['r (nm)', 'g_r'])

In [ ]:
import matplotlib.ticker as ticker
fig, ax = plt.subplots(1,1)
ax.plot(df['r (nm)'], df['g_r'], label='HEAD_1_2 - TAIL')
ax.axhline(1, ls='--', color='black')
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.set(xlabel='r (nm)', ylabel='g(r)')
ax.legend()

first peak corresponds to maybe intramolecular interaction or noise and the second peak corresponds to intermolecular interaction

## micelle structure characterization

In [82]:
import MDAnalysis as mda
import numpy as np
from sklearn.cluster import DBSCAN
import numpy as np

In [ ]:
u = mda.Universe(f'{dir}/nvt_prod.gro', f'{dir}/nvt_prod.xtc')

surf = u.select_atoms("not resname W")  # selecting surfactants

# atom groups corresponding to head and tails of the surfactant
HEAD_3_4 = ['B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'B13', 'B14', 'B15']
HEAD_1_2 = ['B19', 'B20', 'B21', 'B22', 'B24', 'B251', 'B26', 'B27']
TAIL = ['C32', 'C33', 'C0', 'C1']

rdf_bins = np.linspace(0, 50, 100)
rdf_hist_HEAD_3_4 = np.zeros(len(rdf_bins)-1)
rdf_hist_HEAD_1_2 = np.zeros(len(rdf_bins)-1)
rdf_hist_TAIL = np.zeros(len(rdf_bins)-1)

for ts in u.trajectory[-1::]:
    coords = surf.positions
    names = surf.names
    # clustering with cutoff of 2nm or 20A, as indicative from rdf analysis
    clustering = DBSCAN(eps=22, min_samples=68).fit(coords) # eps is the cutoff. Since MDAnalysis converts data to Angstorms, account for units. min_samples keeping it at 2*atom_count of surfactant
    labels = clustering.labels_ # assigns label each atom of the surfactant 

    # navigate through each cluster
    for lbl in set(labels):
        if(lbl == -1):
            continue
        else:
            cluster_indices = np.where(labels == lbl)[0]
            cluster_coords = coords[cluster_indices]
            # calculate center of mass coordinates of the selected cluster
            com = cluster_coords.mean(axis=0)
            
            # group based on the atom type as head/tail
            HEAD_3_4_indices = []
            HEAD_1_2_indices = []
            TAIL_indices = []
            for i, grp in enumerate(names[cluster_indices]):
                # grp = str(atr_grp)
                if grp in HEAD_3_4:
                    HEAD_3_4_indices.append(i)
                elif grp in HEAD_1_2:
                    HEAD_1_2_indices.append(i)
                elif grp in TAIL:
                    TAIL_indices.append(i)

            # HEAD_3_4 rdf
            rho = len(HEAD_3_4_indices) / V_total
            HEAD_3_4_cluster_coordinates = cluster_coords[HEAD_3_4_indices]
            distances = np.linalg.norm(HEAD_3_4_cluster_coordinates - com, axis=1)
            hist, _ = np.histogram(distances, bins=rdf_bins)
            rdf_hist_HEAD_3_4 += hist

            # HEAD_1_2 rdf
            rho = len(HEAD_1_2_indices) / V_total
            HEAD_1_2_cluster_coordinates = cluster_coords[HEAD_1_2_indices]
            distances = np.linalg.norm(HEAD_1_2_cluster_coordinates - com, axis=1)
            hist, _ = np.histogram(distances, bins=rdf_bins)
            rdf_hist_HEAD_1_2 += hist

            # TAIL rdf
            rho = len(TAIL_indices) / V_total
            TAIL_cluster_coordinates = cluster_coords[TAIL_indices]
            distances = np.linalg.norm(TAIL_cluster_coordinates - com, axis=1)
            hist, _ = np.histogram(distances, bins=rdf_bins)
            rdf_hist_TAIL += hist


In [ ]:
# rdf_hist_HEAD_1_2
rdf_hist_TAIL

In [135]:
V_total = 230**3 # in Ang^3
rho = len(surf.names) / V_total
r = 0.5 * (rdf_bins[1:] + rdf_bins[:-1])
dr = rdf_bins[1] - rdf_bins[0]   
shell_volume = 4*np.pi*r**2*dr
N_rdf_hist_HEAD_3_4 = rdf_hist_HEAD_3_4 / shell_volume / rho  # normalization
N_rdf_hist_HEAD_1_2 = rdf_hist_HEAD_1_2 / shell_volume / rho  # normalization
N_rdf_hist_TAIL = rdf_hist_TAIL / shell_volume / rho  # normalization

In [ ]:
import matplotlib.ticker as ticker
fig, ax = plt.subplots(1,1)
r_centers = 0.5 * (rdf_bins[1:] + rdf_bins[:-1]) # convert to nm?
ax.plot(r_centers, N_rdf_hist_HEAD_3_4, label='HEAD_3_4')
ax.plot(r_centers, N_rdf_hist_HEAD_1_2, label='HEAD_1_2')
# ax.plot(r_centers, N_rdf_hist_TAIL, label='TAIL')
ax.axhline(1, ls='--', color='black')
ax.legend()
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))

### number of clusters (from manual clustering)

In [92]:
u = mda.Universe(f'{dir}/nvt_prod.gro', f'{dir}/nvt_prod.xtc')

surf = u.select_atoms("not resname W")  # selecting surfactants

In [106]:
number_clusters = [] # list to hold number of clusters at each time instance
for ts in u.trajectory[::10]:
    coords = surf.positions
    # clustering with cutoff of 2nm or 20A, as indicative from rdf analysis
    clustering = DBSCAN(eps=22, min_samples=68).fit(coords) # eps is the cutoff. Since MDAnalysis converts data to Angstorms, account for units. min_samples keeping it at 2*atom_count of surfactant
    labels = clustering.labels_ # assigns label each atom of the surfactant 
    # print(f"no of unique labels at {ts.time}ps: {len(set(labels))-1}") # ignoring noise
    number_clusters.append([ts.time, len(set(labels))-1]) # appending time and number of clusters to python list
number_clusters  = np.array(number_clusters)

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(number_clusters[:,0], number_clusters[:,1])
ax.set(xlabel='Time (ps)', ylabel='No. of Clusters')


### number of clusters (from gromacs)

In [49]:
file = 'properties/nmicelles.xvg'
# Use StringIO to treat the filtered lines as a file for read_csv
removed_comments = filter_comments(f'{dir}/{file}')
file_removed_comments = StringIO('\n'.join(removed_comments))
df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=['time (ps)', 'clusters'])

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(df['time (ps)'], df['clusters'])

# Visualization

In [62]:
import MDAnalysis as mda
import nglview as nv

u = mda.Universe(f'{dir}/nvt_prod.gro', f'{dir}/nvt_prod.xtc')
view = nv.show_mdanalysis(u)

In [ ]:
from nglview.contrib.movie import MovieMaker
output_filename = f'{dir}/simulation_movie.gif'

# Create the MovieMaker instance from your view
movie = MovieMaker(view, output=output_filename, timeout=60, start=0, step=100) 

# Render and save the GIF
movie.make()